In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# import

In [ ]:
data_path = os.getenv('DATA_PATH')
comments = pd.read_csv(f'{data_path}/comments_with_label_2.csv')
comments.head(1)

# preprocess 

In [ ]:
df_train = comments[(comments['sentiment'].isin(['Negative', 'Positive', 'Neutral']) &
                     (comments['cleaned_comment'].notna())
                                    )][['sentiment', 'cleaned_comment', 'star', 'description']]

In [ ]:
cnt = dict(df_train['sentiment'].value_counts())
cnt

In [ ]:
labels = list(cnt.keys())
sizes = list(cnt.values())
fig = px.histogram(x=labels, y=sizes)
fig.show()

In [ ]:
fig = go.Figure()

groupby_rate = df_train.groupby('star')['star'].count()

fig.add_trace(go.Bar(
    x=list(sorted(groupby_rate.index)),
    y=groupby_rate.tolist(),
    text=groupby_rate.tolist(),
    textposition='auto'
))

fig.update_layout(
    title_text='Distribution of star within comments',
    xaxis_title_text='Rate',
    yaxis_title_text='Frequency',
    bargap=0.2,
    bargroupgap=0.2)

fig.show()

In [ ]:
def word_tokenize(x):
    return x.split(' ')

In [ ]:
df_train['len_comment_by_word'] = df_train['description'].map(lambda x: len(word_tokenize(x)))
min_max_len = df_train["len_comment_by_word"].min(), df_train["len_comment_by_word"].max()
print(f'Min: {min_max_len[0]} \tMax: {min_max_len[1]}')

fig = go.Figure()

groupby_rate = df_train.groupby('len_comment_by_word')['len_comment_by_word'].count()

fig.add_trace(go.Bar(
    x=list(sorted(groupby_rate.index)),
    y=groupby_rate.tolist(),
    text=groupby_rate.tolist(),
    textposition='auto'
))

fig.update_layout(
    title_text='Distribution of comment words within comments',
    xaxis_title_text='Rate',
    yaxis_title_text='Frequency',
    bargap=0.2,
    bargroupgap=0.2)

fig.show()

In [ ]:
df_train['len_comment_after_preprocess_by_word'] = df_train['cleaned_comment'].apply(lambda x: len(word_tokenize(x)))
min_max_len = df_train["len_comment_after_preprocess_by_word"].min(), df_train["len_comment_after_preprocess_by_word"].max()
print(f'Min: {min_max_len[0]} \tMax: {min_max_len[1]}')

fig = go.Figure()

groupby_rate = df_train.groupby('len_comment_after_preprocess_by_word')['len_comment_after_preprocess_by_word'].count()

fig.add_trace(go.Bar(
    x=list(sorted(groupby_rate.index)),
    y=groupby_rate.tolist(),
    text=groupby_rate.tolist(),
    textposition='auto'
))

fig.update_layout(
    title_text='Distribution of comment words after preprocess within comments',
    xaxis_title_text='Rate',
    yaxis_title_text='Frequency',
    bargap=0.2,
    bargroupgap=0.2)

fig.show()

In [ ]:
df_train = df_train.drop(df_train[df_train['cleaned_comment'].apply(lambda x: len(word_tokenize(x)) > 1)==False].index)

In [ ]:
fig = go.Figure()

groupby_rate = df_train.groupby('len_comment_after_preprocess_by_word')['len_comment_after_preprocess_by_word'].count()

fig.add_trace(go.Bar(
    x=list(sorted(groupby_rate.index)),
    y=groupby_rate.tolist(),
    text=groupby_rate.tolist(),
    textposition='auto'
))

fig.update_layout(
    title_text='Distribution of comment words after preprocess within comments',
    xaxis_title_text='Rate',
    yaxis_title_text='Frequency',
    bargap=0.2,
    bargroupgap=0.2)

fig.show()

In [ ]:
data = df_train.reset_index()[['cleaned_comment', 'sentiment']]
data.columns = ['comment', 'label']
data.shape

In [ ]:
fig = go.Figure()

groupby_label = data.groupby('label')['label'].count()

fig.add_trace(go.Bar(
    x=list(sorted(groupby_label.index)),
    y=groupby_label.tolist(),
    text=groupby_label.tolist(),
    textposition='auto'
))

fig.update_layout(
    title_text='Distribution of label within comments [DATA]',
    xaxis_title_text='Label',
    yaxis_title_text='Frequency',
    bargap=0.2,
    bargroupgap=0.2)

fig.show()

In [ ]:
data['label'] = data['label'].map({
    'Negative': 0,
    "Neutral": 1,
    'Positive': 2,
})

In [ ]:
def tokenize(text):
    return word_tokenize(text)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data['comment'], data['label'], test_size=0.1, random_state=42)

# models

## classic ML

In [ ]:
naive_bayes = Pipeline([('vect', CountVectorizer(tokenizer=tokenize,
                                              analyzer='word', ngram_range=(1, 2), min_df=1, lowercase=False)),
                     ('tfidf', TfidfTransformer(sublinear_tf=True)),
                     ('clf', MultinomialNB())])
naive_bayes = naive_bayes.fit(X_train, y_train)
naive_score = naive_bayes.score(X_test, y_test)
print('Naive Bayes Model: ', naive_score)
predict_nb = naive_bayes.predict(X_test)
print("f1 score : ", f1_score(predict_nb, y_test, average='weighted'))
print(classification_report(predict_nb, y_test))

In [ ]:
# SGD (Stochastic Gradient Descent) Model
sgd = Pipeline([('vect', CountVectorizer(tokenizer=tokenize,
                                                  analyzer='word', ngram_range=(1, 2), min_df=1, lowercase=False)),
                         ('tfidf', TfidfTransformer(sublinear_tf=True)),
                         ('clf-svm', SGDClassifier(loss='hinge', penalty='l2',
                                                   alpha=1e-3, max_iter=5))])
sgd = sgd.fit(X_train, y_train)
sgd_score = sgd.score(X_test, y_test)
print('SGD Model: ', sgd_score)
predict_sgd = sgd.predict(X_test)
print("f1 score : ", f1_score(predict_nb, y_test, average='weighted'))
print(classification_report(predict_sgd, y_test))

In [ ]:
# Linear Support Vector Machine Model
svm = Pipeline([
    ('vect', CountVectorizer(tokenizer=tokenize, analyzer='word', ngram_range=(1, 3), min_df=1, lowercase=False)),
    ('tfidf', TfidfTransformer(sublinear_tf=True)),
    ('clf-svm', LinearSVC(loss='hinge', penalty='l2', max_iter=5))
                ])

svm = svm.fit(X_train, y_train)
linear_svc_score = svm.score(X_test, y_test)
print('Linear SVC Model: ', linear_svc_score)
predict_svm = svm.predict(X_test)
print( "f1 score : ", f1_score(predict_svm, y_test, average='weighted'))
print(classification_report(predict_svm, y_test))

## DL

In [ ]:
label_mapping = {
    0 : 'Negative',
    1 : "Neutral",
    2 : 'Positive'
}

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
print(torch.cuda.is_available())  # باید True برگرداند
print(torch.cuda.get_device_name(0))  # نمایش نام کارت گرافیک

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = Tokenizer(num_words=5000, lower=True, split=' ')
tokenizer.fit_on_texts(X_train.values)
X_train_seq = tokenizer.texts_to_sequences(X_train.values)
X_test_seq = tokenizer.texts_to_sequences(X_test.values)

X_train_pad = pad_sequences(X_train_seq)
X_test_pad = pad_sequences(X_test_seq, maxlen=X_train_pad.shape[1])

X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_output):
        attn_weights = torch.tanh(self.attention(lstm_output)).squeeze(-1)
        attn_weights = F.softmax(attn_weights, dim=1).unsqueeze(1)
        context_vector = torch.bmm(attn_weights, lstm_output).squeeze(1)
        return context_vector

class AdvancedBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout):
        super(AdvancedBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=4, bidirectional=True, batch_first=True, dropout=dropout)
        self.attention = Attention(hidden_dim)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim * 2)
        self.fc1 = nn.Linear(hidden_dim * 2, 512)
        self.batch_norm2 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.batch_norm3 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc4 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.attention(x) 
        x = self.batch_norm1(x)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.batch_norm2(x)
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.batch_norm3(x)
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x


In [ ]:
model = AdvancedBiLSTM(vocab_size=5000, embed_dim=300, hidden_dim=256, output_dim=3, dropout=0.5)

pretrained_embeddings = torch.rand(5000, 300)  
model.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.CyclicLR(optimizer, base_lr=1e-5, max_lr=1e-3, step_size_up=2000, mode='triangular2')
model = model.to(device)

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)  
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

model.eval()
y_pred, y_true = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        y_pred.extend(preds.cpu().numpy())  #
        y_true.extend(labels.cpu().numpy())  
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

In [ ]:
import json
model_path = '../models/sentiment_analysis_model_on_30K_comment.pth'
torch.save(model.state_dict(), model_path)

model_details = {
    "model_name": "sentiment_analysis_model_on_30K_comment",
    "accuracy": (np.array(y_true) == np.array(y_pred)).mean(),
    "f1_score": f1_score(y_true, y_pred, average='weighted'),
    "classification_report": classification_report(y_true, y_pred, output_dict=True)
}

with open('../models/sentiment_analysis_model_on_30K_comment_details.json', 'w') as f:
    json.dump(model_details, f)

In [ ]:
model_path = '../models/sentiment_analysis_model_on_30K_comment_full.pth'
torch.save(model, model_path)

In [ ]:
sample_comments = comments[comments['cleaned_comment'].notna()].sample(100)
star = sample_comments['star']
sample_comments = sample_comments['cleaned_comment']

sample_sequences = tokenizer.texts_to_sequences(sample_comments)

sample_padded = pad_sequences(sample_sequences, maxlen=X_train_pad.shape[1])
sample_tensor = torch.tensor(sample_padded, dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    sample_predictions = model(sample_tensor)
    sample_pred_classes = torch.argmax(sample_predictions, dim=1).cpu().numpy()

predicted_sentiments = [label_mapping[pred] for pred in sample_pred_classes]

results_df = pd.DataFrame({
    'comment': sample_comments.values,
    'predict': predicted_sentiments,
    'star': star.values
})

for i in results_df.values:
    print(i[2], i[1], i[0])